In [ ]:
#Clean up dose_unit, route, dose_cbm columns
import polars as pl

lazy_df = pl.scan_parquet("/kaggle/input/datasets/maxthacker/fda-faers-ascii-dataset/drug_normalized_clean.parquet")

lazy_df = lazy_df.with_columns(
    pl.all()
    .str.strip_chars()
    .str.to_uppercase()
)
lazy_df = lazy_df.with_columns(
    pl.col('dose_unit').str.replace_all(" ", "")
)


lazy_df = lazy_df.with_columns(
    pl.col('dose_unit')
    .str.replace_all(r' ','')
    .alias('dose_unit') 
)

updated_df = (
    lazy_df
    .with_columns([
        # ── dose_unit ─────────────────────────────────────────────
        pl.when(pl.col("dose_unit") == "MCG")
        .then(pl.lit("UG"))
        .when(pl.col("dose_unit").is_in(['GM','GRAM']))
        .then(pl.lit("G"))
        .when(pl.col("dose_unit") == "MG.")
        .then(pl.lit("MG"))
        .when(pl.col("dose_unit") == "UI")
        .then(pl.lit("IU"))
        .when(pl.col("dose_unit").is_in(['MG/M2','MG/M^2']))
        .then(pl.lit("MG/M**2"))
        .otherwise(pl.col("dose_unit"))
        .alias("dose_unit"),

        # ── route ─────────────────────────────────────────────────
        pl.when(pl.col("route").is_in(["UNK", "U", "NS", ""]))
        .then(pl.lit("UNKNOWN"))
        .otherwise(pl.col("route"))
        .alias("route"),

        # ── dose_vbm ──────────────────────────────────────────────
        pl.when(pl.col("dose_vbm").is_in(["UNK", "U", "NS", ""]))
        .then(pl.lit("UNKNOWN"))
        .otherwise(pl.col("dose_vbm"))
        .alias("dose_vbm"),

        # ── dose_form ──────────────────────────────────────────────
        pl.when(pl.col("dose_form").is_in(["UNK", "U", "NS", "",'?','??','UNKNOWN FORMULATION','FORMULATION UNKNOWN','UNSPECIFIED']))
        .then(pl.lit("UNKNOWN"))
        .when(pl.col("dose_form").is_in(['TAB','TABLETS','TAB.','FILM-COATED TABLET']))
        .then(pl.lit("TABLET"))
        .when(pl.col("dose_form").is_in(['CAP','CAPS','CAPSULES','CAPSULE, HARD','SOFTGEL CAPSULE']))
        .then(pl.lit("CAPSULE"))
        .when(pl.col("dose_form").is_in(['SOL','SOLN','ORAL SOLUTION']))
        .then(pl.lit("SOLUTION"))
        .when(pl.col("dose_form").is_in(['INJ','INJECTIONS','INJECTABLE','SOLUTION FOR INJECTION']))
        .then(pl.lit("INJECTION"))
        .when(pl.col("dose_form").is_in(['SUPS']))
        .then(pl.lit("SUSPENSION"))
        .when(pl.col("dose_form").is_in(['PWD','PWDR']))
        .then(pl.lit("POWDER"))
        .otherwise(pl.col("dose_form"))
        .alias("dose_form"),

        # ── cum_dose_unit  ─────────────────────────────────────────────
        pl.when(pl.col("cum_dose_unit") == "GM")
        .then(pl.lit("G"))
        .when(pl.col("cum_dose_unit") == "MG/M2")
        .then(pl.lit("MG/M**2"))
        .when(pl.col("cum_dose_unit") == "UG/M2")
        .then(pl.lit("UG/M**2"))
        .otherwise(pl.col("cum_dose_unit"))
        .alias("cum_dose_unit"),

    ])
)

In [ ]:
# Dedupe main demo file from 8 column dedupe
import polars as pl

lazy_8_cols = pl.scan_parquet("/kaggle/input/datasets/maxthacker/fda-faers-ascii-dataset/final_demo_dedup.parquet")

lazy_8_cols = lazy_8_cols.select(pl.col('primaryid'))

lazy_demo_df = pl.scan_parquet("/kaggle/input/datasets/maxthacker/fda-faers-ascii-dataset/Merged/demo.parquet")

lazy_demo_df = lazy_demo_df.with_columns(
    pl.col("primaryid").cast(pl.Int64)
)

lf_filtered = lazy_demo_df.join(lazy_8_cols, on='primaryid', how="semi")

lf_filtered.sink_parquet("/kaggle/working/demo_dedup_final.parquet")


In [ ]:
# clean up deduped demo file
import polars as pl
pl.Config.set_tbl_rows(40)

lazy_df = pl.scan_parquet("/kaggle/working/demo_dedup_final.parquet")

lazy_df = lazy_df.with_columns(
    pl.col(['sex','age','age_cod','wt','wt_cod'])
    .str.strip_chars()
    .str.to_uppercase()
)

# make binary sex column
updated_df = (
    lazy_df
    .with_columns([
        # ── Binary Sex ─────────────────────────────────────────────
        pl.when(pl.col("sex") == "F")
        .then(1)
        .when(pl.col("sex") == "M")
        .then(0)
        .when(pl.col("sex").is_in(['P','I','T','UNK']))
        .then(None)
        .otherwise(pl.col("sex"))
        .alias("sex_bin")
        .cast(pl.Int8),
    ])
)
#Age dict
age_to_years = {
    "YR":  1.0,
    "DEC": 10.0,
    "MON": 1/12,
    "WK":  1/52.18,
    "DY":  1/365.25,
    "HR":  1/8766.0,
    "MIN": 1/525960.0,
    "SEC": 1/31557600.0,
}

#convert age with age con to total years
updated_df = updated_df.with_columns(
    (pl.col("age").cast(pl.Float64, strict = False) *
     pl.col("age_cod").replace_strict(age_to_years, default=None))
    .alias("age_years")
)
# Remove odd age
updated_df = updated_df.with_columns(
    pl.when(pl.col("age_years").is_between(0, 120))
      .then(pl.col("age_years"))
      .otherwise(None)
      .alias("age_years")
)

# Weight dict
wt_to_kg = {
    "KG":  1.0,
    "KGS": 1.0,
    "LBS": 0.453592,
    "IB":  0.453592,
    "GMS": 0.001,
}

#coerce to float
updated_df = updated_df.with_columns(
    pl.col("wt")
      .str.replace(",", ".")       
      .str.replace_all(r"[^\d.]", "") 
      .cast(pl.Float64, strict=False) 
)

# Weight conversion to kg
updated_df = updated_df.with_columns(
    (pl.col("wt").cast(pl.Float64) *
     pl.col("wt_cod").replace_strict(wt_to_kg, default=None))
    .alias("wt_kg")
)

# Remove odd weight values
updated_df = updated_df.with_columns(
    pl.when(pl.col("wt_kg").is_between(1.0, 300.0))
      .then(pl.col("wt_kg"))
      .otherwise(None)
      .alias("wt_kg")
)

updated_df.sink_parquet("/kaggle/working/demo_dedup_clean.parquet")


In [ ]:
# Select columns from Drug and Demo, then merge.

import polars as pl

lazy_demo = pl.scan_parquet("/kaggle/working/demo_dedup_clean.parquet")

demo_df = lazy_demo.select([pl.col('primaryid').cast(pl.Int64),
                            'sex',
                            'sex_bin',
                            'age_years',
                            'wt_kg'])

# print(demo_df)

lazy_drug = pl.scan_parquet('/kaggle/input/datasets/maxthacker/fda-faers-ascii-dataset/drug_normalized_clean.parquet')

drug_df = lazy_drug.select([pl.col('primaryid').cast(pl.Int64),
                            pl.col('caseid').cast(pl.Int64),
                            'role_cod',
                            'rx_name',
                            'dose_amt',
                            'dose_unit',
                            'dose_form'])

# print(drug_df)

(drug_df.join(demo_df, on= 'primaryid',how='left')
    .sink_parquet('/kaggle/working/drug_demo_merge.parquet'))


In [ ]:
#Select outc then merge on drug_demo 
import polars as pl

lazy_outc = pl.scan_parquet("/kaggle/input/datasets/maxthacker/fda-faers-ascii-dataset/Merged/outc.parquet")

outcome_cols = [
    "outcome_death",
    "outcome_life_threatening",
    "outcome_hospitalisation",
    "outcome_disability",
    "outcome_congenital_anomaly",
    "outcome_intervention_required",
    "outcome_other_serious",
]

outc = (
    lazy_outc
    .select([
        pl.col("primaryid").cast(pl.Int64),
        "outc_cod"
    ])
    .with_columns([
        (pl.col("outc_cod") == "DE").alias("outcome_death"),
        (pl.col("outc_cod") == "LT").alias("outcome_life_threatening"),
        (pl.col("outc_cod") == "HO").alias("outcome_hospitalisation"),
        (pl.col("outc_cod") == "DS").alias("outcome_disability"),
        (pl.col("outc_cod") == "CA").alias("outcome_congenital_anomaly"),
        (pl.col("outc_cod") == "RI").alias("outcome_intervention_required"),
        (pl.col("outc_cod") == "OT").alias("outcome_other_serious"),
    ])
    .group_by("primaryid")
    .agg([pl.col(c).max() for c in outcome_cols])  
)

(pl.scan_parquet("/kaggle/working/drug_demo_merge.parquet")
     .join(outc, on="primaryid", how="left")
     .sink_parquet("drug_demo_outc_merge.parquet"))

In [ ]:
#Select reac then merge on drug_demo 
import polars as pl

lazy_reac = pl.scan_parquet("/kaggle/input/datasets/maxthacker/fda-faers-ascii-dataset/Merged/reac.parquet")

# print(lazy_reac)

reac = (
    lazy_reac
    .select([
        pl.col("primaryid").cast(pl.Int64),
        "pt"
    ])
    .group_by("primaryid")
    .agg(pl.col("pt").alias("reactions"))  
)

(pl.scan_parquet("/kaggle/working/drug_demo_outc_merge.parquet")
     .join(reac, on="primaryid", how="left")
     .sink_parquet("/kaggle/working/drug_demo_outc_reac_merge.parquet"))